## Import Dependencies

In [5]:
from transformers import AutoTokenizer
import polars as pl
from datasets import Dataset

## Import Data

In [6]:
train_path = "/content/complete_train.parquet"
test_path = "/content/complete_test.parquet"

In [7]:
train_df = pl.scan_parquet(train_path)
test_df = pl.scan_parquet(test_path)

In [8]:
test_df.collect()

id,asym_id,amino_acid,index,strand_id,secondary_structure,rcsb_id,sequence,length
str,str,str,i64,str,str,str,str,i64
"""3VKW""","""A""","""SER""",1,"""A""",""" ""","""3VKW_1""","""SYTRSEEIESLEQFHMATASSLIHKQMCSI…",446
"""3VKW""","""A""","""TYR""",2,"""A""",""" ""","""3VKW_1""","""SYTRSEEIESLEQFHMATASSLIHKQMCSI…",446
"""3VKW""","""A""","""THR""",3,"""A""",""" ""","""3VKW_1""","""SYTRSEEIESLEQFHMATASSLIHKQMCSI…",446
"""3VKW""","""A""","""ARG""",4,"""A""",""" ""","""3VKW_1""","""SYTRSEEIESLEQFHMATASSLIHKQMCSI…",446
"""3VKW""","""A""","""SER""",5,"""A""",""" ""","""3VKW_1""","""SYTRSEEIESLEQFHMATASSLIHKQMCSI…",446
…,…,…,…,…,…,…,…,…
"""5XVT""","""A""","""LEU""",693,"""A""","""T""","""5XVT_1""","""MGSSHHHHHHSSGLVPRGSHMSSVDQKAIS…",697
"""5XVT""","""A""","""ARG""",694,"""A""",""".""","""5XVT_1""","""MGSSHHHHHHSSGLVPRGSHMSSVDQKAIS…",697
"""5XVT""","""A""","""SER""",695,"""A""",""".""","""5XVT_1""","""MGSSHHHHHHSSGLVPRGSHMSSVDQKAIS…",697


## Process Data

### Add Labels

In [9]:
train_df = (train_df
            .with_columns(label = pl.col('secondary_structure').rank(method='dense') - 1)
              .cast({'label': pl.Int64}))

test_df = (test_df
            .with_columns(label = pl.col('secondary_structure').rank(method='dense') - 1)
              .cast({'label': pl.Int64}))


### Removing Duplicates on Index
Some indices have two amino acids, so remove the one without a secondary structure (label = 0)

In [10]:
train_removed = (
    train_df.sort(['id', 'asym_id', 'index', 'label'])
            .group_by(['id', 'asym_id', 'index'])
            .agg(['amino_acid', 'label', 'secondary_structure'])
            .filter((pl.col('amino_acid').list.len() > 1))
            .with_columns(
                amino_acid = pl.col('amino_acid').list.slice(1),
                label = pl.col('label').list.slice(1),
                secondary_structure = pl.col('secondary_structure').list.slice(1),
            )
            .explode(['amino_acid', 'label', 'secondary_structure'])
)

train_df = (
    train_df.join(train_removed, on=['id', 'asym_id', 'index'], how='left')
            .with_columns(
                          amino_acid = pl.when(~(pl.col('amino_acid_right').is_null()))
                                        .then(pl.col('amino_acid_right'))
                                        .otherwise(pl.col('amino_acid')),
                         label = pl.when(~(pl.col('label_right').is_null()))
                                        .then(pl.col('label_right'))
                                        .otherwise(pl.col('label')),
                          secondary_structure = pl.when(~(pl.col('secondary_structure_right').is_null()))
                                        .then(pl.col('secondary_structure_right'))
                                        .otherwise(pl.col('secondary_structure'))
            )
            .drop(['amino_acid_right', 'label_right', 'secondary_structure_right'])
            .unique(['id', 'asym_id', 'index'])
            .sort(['id', 'asym_id', 'index'])
)


In [11]:
test_removed = (
    test_df.sort(['id', 'asym_id', 'index', 'label'])
            .group_by(['id', 'asym_id', 'index'])
            .agg(['amino_acid', 'label', 'secondary_structure'])
            .filter((pl.col('amino_acid').list.len() > 1))
            .with_columns(
                amino_acid = pl.col('amino_acid').list.slice(1),
                label = pl.col('label').list.slice(1),
                secondary_structure = pl.col('secondary_structure').list.slice(1),
            )
            .explode(['amino_acid', 'label', 'secondary_structure'])
)

test_df = (
    test_df.join(test_removed, on=['id', 'asym_id', 'index'], how='left')
            .with_columns(
                          amino_acid = pl.when(~(pl.col('amino_acid_right').is_null()))
                                        .then(pl.col('amino_acid_right'))
                                        .otherwise(pl.col('amino_acid')),
                         label = pl.when(~(pl.col('label_right').is_null()))
                                        .then(pl.col('label_right'))
                                        .otherwise(pl.col('label')),
                          secondary_structure = pl.when(~(pl.col('secondary_structure_right').is_null()))
                                        .then(pl.col('secondary_structure_right'))
                                        .otherwise(pl.col('secondary_structure'))
            )
            .drop(['amino_acid_right', 'label_right', 'secondary_structure_right'])
            .unique(['id', 'asym_id', 'index'])
            .sort(['id', 'asym_id', 'index'])
)


In [12]:
test_df.collect()

id,asym_id,amino_acid,index,strand_id,secondary_structure,rcsb_id,sequence,length,label
str,str,str,i64,str,str,str,str,i64,i64
"""1A1X""","""A""","""GLY""",1,"""A""",""" ""","""1A1X_1""","""GSAGEDVGAPPDHLWVHQEGIYRDEYQRTW…",108,0
"""1A1X""","""A""","""SER""",2,"""A""",""" ""","""1A1X_1""","""GSAGEDVGAPPDHLWVHQEGIYRDEYQRTW…",108,0
"""1A1X""","""A""","""ALA""",3,"""A""",""".""","""1A1X_1""","""GSAGEDVGAPPDHLWVHQEGIYRDEYQRTW…",108,1
"""1A1X""","""A""","""GLY""",4,"""A""",""".""","""1A1X_1""","""GSAGEDVGAPPDHLWVHQEGIYRDEYQRTW…",108,1
"""1A1X""","""A""","""GLU""",5,"""A""",""".""","""1A1X_1""","""GSAGEDVGAPPDHLWVHQEGIYRDEYQRTW…",108,1
…,…,…,…,…,…,…,…,…,…
"""7ODC""","""A""","""ILE""",420,"""A""",""" ""","""7ODC_1""","""MSSFTKDEFDCHILDEGFTAKDILDQKINE…",424,0
"""7ODC""","""A""","""GLN""",421,"""A""",""" ""","""7ODC_1""","""MSSFTKDEFDCHILDEGFTAKDILDQKINE…",424,0
"""7ODC""","""A""","""SER""",422,"""A""",""" ""","""7ODC_1""","""MSSFTKDEFDCHILDEGFTAKDILDQKINE…",424,0


#### Remove Duplicates on Asym ID
Some sequences are exactly the same and have the same secondary structures, so choose only one Asym ID (first ID)

In [13]:
train_df = (

  (train_df.sort(['id', 'asym_id', 'index'])
              .group_by(['id', 'asym_id', 'sequence'])
                .agg(['secondary_structure', 'label', 'index'])

  ).sort('asym_id')
    .group_by(['id', 'sequence', 'secondary_structure', 'label', 'index'])
      .agg(['asym_id'])
        .with_columns(asym_id = pl.col('asym_id').list.slice(0,1))
          .explode(['asym_id'])
            .explode(['secondary_structure', 'label', 'index'])

)

test_df = (

  (test_df.sort(['id', 'asym_id', 'index'])
              .group_by(['id', 'asym_id', 'sequence'])
                .agg(['secondary_structure', 'label', 'index'])

  ).sort('asym_id')
    .group_by(['id', 'sequence', 'secondary_structure', 'label', 'index'])
      .agg(['asym_id'])
        .with_columns(asym_id = pl.col('asym_id').list.slice(0,1))
          .explode(['asym_id'])
            .explode(['secondary_structure', 'label', 'index'])

)



In [14]:
test_df.collect()

id,sequence,secondary_structure,label,index,asym_id
str,str,str,i64,i64,str
"""3BL9""","""SAPVRLPFSGFRLQKVLRESARDKIIFLHG…",""" """,0,1,"""B"""
"""3BL9""","""SAPVRLPFSGFRLQKVLRESARDKIIFLHG…",""" """,0,2,"""B"""
"""3BL9""","""SAPVRLPFSGFRLQKVLRESARDKIIFLHG…",""" """,0,3,"""B"""
"""3BL9""","""SAPVRLPFSGFRLQKVLRESARDKIIFLHG…",""".""",1,4,"""B"""
"""3BL9""","""SAPVRLPFSGFRLQKVLRESARDKIIFLHG…",""".""",1,5,"""B"""
…,…,…,…,…,…
"""4ZKY""","""GMAEFDAVTAFADAPAAVLSTLNADGAPHL…","""E""",3,139,"""B"""
"""4ZKY""","""GMAEFDAVTAFADAPAAVLSTLNADGAPHL…","""E""",3,140,"""B"""
"""4ZKY""","""GMAEFDAVTAFADAPAAVLSTLNADGAPHL…","""E""",3,141,"""B"""


### Prepare for Tokenization

In [15]:
train_agg = (train_df
              .select(['id', 'asym_id', 'sequence', 'index', 'secondary_structure','label'])
              .sort(['id', 'asym_id', 'index'])
              .group_by(['id', 'asym_id', 'sequence'])
              .agg(['secondary_structure','index', 'label'])
            )

test_agg = (test_df
              .select(['id', 'asym_id', 'sequence', 'index', 'secondary_structure', 'label'])
              .sort(['id', 'asym_id', 'index'])
              .group_by(['id', 'asym_id', 'sequence'])
              .agg(['secondary_structure','index', 'label'])
            )

In [16]:
train_ds = Dataset.from_polars(train_agg.collect())
test_ds = Dataset.from_polars(test_agg.collect())

### Tokenization

In [17]:
tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t6_8M_UR50D")

def tokenize_and_label(examples):
  tokenized_inputs = tokenizer(examples["sequence"],
                                return_tensors="pt",
                                add_special_tokens=False,
                                padding=False)
  tokenized_inputs["input_ids"] = tokenized_inputs["input_ids"][0]
  tokenized_inputs["attention_mask"] = tokenized_inputs["attention_mask"][0]
  return tokenized_inputs

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [18]:
tokenized_train = train_ds.map(tokenize_and_label, batched=False)
tokenized_test = test_ds.map(tokenize_and_label, batched=False)

Map:   0%|          | 0/11425 [00:00<?, ? examples/s]

Map:   0%|          | 0/2850 [00:00<?, ? examples/s]

## Output Data

### Dataset to DataFrame

In [19]:
output_train = Dataset.to_polars(tokenized_train).lazy()
output_test = Dataset.to_polars(tokenized_test).lazy()

In [20]:
output_train = (output_train
                .join(train_agg,
                      on=['id', 'asym_id', 'sequence', 'index', 'secondary_structure', 'label'],
                      how='inner')
                .explode(['secondary_structure','index', 'label', 'input_ids', 'attention_mask'])
                )

output_test = (output_test
                .join(test_agg,
                      on=['id', 'asym_id', 'sequence', 'index', 'secondary_structure', 'label'],
                      how='inner')
                .explode(['secondary_structure', 'index', 'label', 'input_ids', 'attention_mask'])
                )


### Write to Parquet

In [21]:
output_train.collect().write_parquet('tokenized_train.parquet')
output_test.collect().write_parquet('tokenized_test.parquet')

#### Replacing Labels and Secondary Structure

In [22]:
output_train_c9 = output_train.with_columns(
                  label=pl.when(pl.col('label')==0)
                                      .then(pl.lit(-100))
                                      .otherwise(pl.col('label')))

output_test_c9 = output_test.with_columns(
                  label=pl.when(pl.col('label')==0)
                                      .then(pl.lit(-100))
                                      .otherwise(pl.col('label')))

In [23]:
output_train_c9.collect().write_parquet('tokenized_train_c9.parquet')
output_test_c9.collect().write_parquet('tokenized_test_c9.parquet')

In [27]:
output_train_c9_v2 = output_train.with_columns(
    secondary_structure=pl.when((pl.col('secondary_structure')==" ") |
                                (pl.col('secondary_structure')=="."))
                                      .then(pl.lit('C'))
                                      .otherwise(pl.col('secondary_structure'))
).with_columns(
    label=pl.col('secondary_structure').rank(method='dense') - 1
).cast({'label': pl.Int64})



output_test_c9_v2 = output_test.with_columns(
    secondary_structure=pl.when((pl.col('secondary_structure')==" ") |
                                (pl.col('secondary_structure')=="."))
                                      .then(pl.lit('C'))
                                      .otherwise(pl.col('secondary_structure'))
).with_columns(
    label=pl.col('secondary_structure').rank(method='dense') - 1
).cast({'label': pl.Int64})


In [28]:
output_test_c9_v2.collect()

id,asym_id,sequence,secondary_structure,index,label,input_ids,attention_mask
str,str,str,str,i64,i64,i32,i8
"""5J1J""","""A""","""GPLGSMKQMGSMHPVQVIAVTGGKGGVGKT…","""C""",1,1,6,1
"""5J1J""","""A""","""GPLGSMKQMGSMHPVQVIAVTGGKGGVGKT…","""C""",2,1,14,1
"""5J1J""","""A""","""GPLGSMKQMGSMHPVQVIAVTGGKGGVGKT…","""C""",3,1,4,1
"""5J1J""","""A""","""GPLGSMKQMGSMHPVQVIAVTGGKGGVGKT…","""C""",4,1,6,1
"""5J1J""","""A""","""GPLGSMKQMGSMHPVQVIAVTGGKGGVGKT…","""C""",5,1,8,1
…,…,…,…,…,…,…,…
"""5OLP""","""B""","""MGSSHHHHHHSSGPQQGLRQYKTVKVKAPF…","""E""",448,2,19,1
"""5OLP""","""B""","""MGSSHHHHHHSSGPQQGLRQYKTVKVKAPF…","""E""",449,2,8,1
"""5OLP""","""B""","""MGSSHHHHHHSSGPQQGLRQYKTVKVKAPF…","""E""",450,2,16,1


In [29]:
output_train_c9_v2.collect().write_parquet('tokenized_train_c9_v2.parquet')
output_test_c9_v2.collect().write_parquet('tokenized_test_c9_v2.parquet')